# Next-Word Prediction using LSTM in PyTorch

This notebook demonstrates building and training a next-word prediction LSTM model in **PyTorch**, using Shakespeare's *Hamlet* as the corpus. This model is migrated from the TensorFlow/Keras equivalent.

### CUDA Diagnostic Check
Run this cell to verify if your dedicated GPU is correctly detected by PyTorch.

In [1]:
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Devices:", torch.cuda.device_count())
    print("Device Name:", torch.cuda.get_device_name(0))

CUDA Available: True
GPU Devices: 1
Device Name: NVIDIA GeForce RTX 4050 Laptop GPU


### 1. Imports

In [2]:
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import re
from collections import Counter
from torchinfo import summary

### 2. Loading the Hamlet Text Corpus

In [ ]:
with open('hamlet.txt', 'r', encoding='utf-8') as file:
    text = file.read().lower()

### 3. Custom Tokenizer implementation
PyTorch does not have an exact built-in equivalent to `tf.keras.preprocessing.text.Tokenizer`. We implement `SimpleTokenizer` to map text to word indexes.

In [ ]:
class SimpleTokenizer:
    def __init__(self):
        self.word_index = {}
        self.index_word = {}
        self.total_words = 0

    def fit_on_texts(self, texts):
        tokens = []
        for t in texts:
            # Extract alphanumeric tokens, similar to Keras default filters
            words = re.findall(r"\b\w+\b", t)
            tokens.extend(words)
        
        # Count frequencies
        counts = Counter(tokens)
        sorted_words = [word for word, count in counts.most_common()]
        
        # Index 0 is reserved for padding (like Keras)
        for idx, word in enumerate(sorted_words, start=1):
            self.word_index[word] = idx
            self.index_word[idx] = word
            
        self.total_words = len(self.word_index) + 1

    def texts_to_sequences(self, texts):
        sequences = []
        for t in texts:
            words = re.findall(r"\b\w+\b", t)
            seq = [self.word_index[word] for word in words if word in self.word_index]
            sequences.append(seq)
        return sequences

In [ ]:
tokenizer = SimpleTokenizer()
tokenizer.fit_on_texts([text])
total_words = tokenizer.total_words
print(f"Total unique words (vocab size): {total_words}")

### 4. Creating N-gram Sequences
We split the text into actual lines using `.splitlines()` to avoid the forward-slash newline bug `/n` present in the original code. This keeps our max sequence length short and efficient.

In [ ]:
input_sequences = []
lines = text.splitlines()

for line in lines:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

print(f"Total N-gram Sequences: {len(input_sequences)}")
max_sequence_len = max([len(x) for x in input_sequences])
print(f"Max Sequence Length: {max_sequence_len}")

### 5. Pre-padding & Data Splitting
We pad the sequences to `max_sequence_len` and extract feature matrix `X` and labels `y`. In PyTorch, we do not need to one-hot encode `y` into a giant dense matrix, saving memory.

In [ ]:
padded_sequences = []
for seq in input_sequences:
    pad_len = max_sequence_len - len(seq)
    padded_seq = [0] * pad_len + seq
    padded_sequences.append(padded_seq)

padded_sequences = np.array(padded_sequences)
X = padded_sequences[:, :-1]
y = padded_sequences[:, -1]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

### 6. Dataset and DataLoader

In [ ]:
class WordPredictionDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)
        
    def __len__(self):
        return len(self.X)
        
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

batch_size = 64
train_dataset = WordPredictionDataset(X_train, y_train)
val_dataset = WordPredictionDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

### 7. Defining the LSTM Model Architecture

In [ ]:
class LSTMWordPredictor(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(LSTMWordPredictor, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, (hn, cn) = self.lstm(embedded)
        # Extract representation at the final sequence timestep
        last_timestep_out = lstm_out[:, -1, :]
        logits = self.fc(last_timestep_out)
        return logits

### GPU Device Setting
Here we define the target `device`. It dynamically selects `cuda` if your GPU is available.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Target training device: {device}")

model = LSTMWordPredictor(vocab_size=total_words, embedding_dim=100, hidden_dim=150).to(device)
summary(model, input_size=(batch_size, max_sequence_len - 1), dtypes=[torch.long])

### 8. Train & Validate Function Declarations
Notice that inside the loader loop, both inputs (`batch_x`) and targets (`batch_y`) are sent to the target `device` (GPU) via `.to(device)` prior to training.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    epoch_loss = 0.0
    correct = 0
    total = 0
    
    for batch_x, batch_y in loader:
        # Send variables to GPU
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item() * batch_x.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(batch_y).sum().item()
        total += batch_y.size(0)
        
    return epoch_loss / total, correct / total

def validate(model, loader, criterion, device):
    model.eval()
    epoch_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch_x, batch_y in loader:
            # Send variables to GPU
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            
            epoch_loss += loss.item() * batch_x.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(batch_y).sum().item()
            total += batch_y.size(0)
            
    return epoch_loss / total, correct / total

### 9. Training the Model

In [ ]:
epochs = 15
print("Starting training loops...")
for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    print(f"Epoch {epoch:02d}/{epochs:02d} | "
          f"Train Loss: {train_loss:.4f} - Train Acc: {train_acc*100:.2f}% | "
          f"Val Loss: {val_loss:.4f} - Val Acc: {val_acc*100:.2f}%")

### 10. Next-Word Generator (Inference)

In [ ]:
def predict_next_words(model, tokenizer, seed_text, next_words=5, max_sequence_len=max_sequence_len):
    model.eval()
    words = seed_text.lower().split()
    
    for _ in range(next_words):
        # Generate token list from seed phrase
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        
        # Keep last (max_sequence_len - 1) tokens
        seq_len = max_sequence_len - 1
        pad_len = seq_len - len(token_list)
        if pad_len > 0:
            token_list = [0] * pad_len + token_list
        else:
            token_list = token_list[-seq_len:]
            
        input_tensor = torch.tensor([token_list], dtype=torch.long).to(device)
        
        with torch.no_grad():
            outputs = model(input_tensor)
            _, predicted_idx = outputs.max(1)
            predicted_idx = predicted_idx.item()
            
        output_word = tokenizer.index_word.get(predicted_idx, "")
        if not output_word:
            break
        
        seed_text += " " + output_word
        
    return seed_text

In [ ]:
seed_phrases = [
    "To be or not to",
    "The tragedie of",
    "Speak to me"
]

for phrase in seed_phrases:
    prediction = predict_next_words(model, tokenizer, phrase, next_words=6)
    print(f"Input: '{phrase}' -> Predicted: '{prediction}'")

### 11. Saving Model Checkpoint

In [ ]:
checkpoint = {
    'model_state_dict': model.state_dict(),
    'word_index': tokenizer.word_index,
    'index_word': tokenizer.index_word,
    'max_sequence_len': max_sequence_len
}
torch.save(checkpoint, 'next_word_lstm.pth')
print("Model saved to next_word_lstm.pth successfully.")